# Notebook 04 — Descomposición Oaxaca-Blinder con bootstrap

Aquí es donde el proyecto rinde su número titular. Tomo los modelos Mincer separados por sexo del notebook 03 y descompongo la brecha mensual en tres bloques:

1. **Horas trabajadas** — cuánto se explica por la diferencia en horas.
2. **Composición** — escolaridad, experiencia, sector, condición de subordinación y entidad.
3. **No explicado (residual)** — la parte que no se justifica por nada observable. Se interpreta como techo de discriminación.

Uso la versión **pooled (Neumark, 1988)** porque es menos sensible al grupo de referencia que las clásicas. Y aplico bootstrap no paramétrico con ~240 réplicas para tener IC al 95%.


In [ ]:
from pathlib import Path
import sys, json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import oaxaca

plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 160, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
})

OUTPUTS = ROOT / "outputs"
df = pd.read_parquet(ROOT / "data" / "processed" / "enoen_2025_4t_mincer.parquet")

df["sector_cat"]    = df["sector"].astype(int).astype(str)
df["ent_cat"]       = df["ent"].astype(int).astype(str)
df["subordinado_i"] = df["subordinado"].astype(int)

formula = ("log_ingreso_mensual_w ~ log_horas_w + anios_escolaridad + experiencia + experiencia2 "
           "+ C(sector_cat) + subordinado_i + C(ent_cat)")

bloques = {
    "horas":       ["log_horas_w"],
    "composicion": ["anios_escolaridad","experiencia","experiencia2",
                    "sector_cat","subordinado_i","ent_cat"],
}
vars_x = ["log_horas_w","anios_escolaridad","experiencia","experiencia2",
          "sector_cat","subordinado_i","ent_cat"]


## 1. Descomposición principal (punto estimado)

Una sola pasada sobre todo el dataset.


In [ ]:
res = oaxaca.descomponer_pooled(df, formula, "fac_tri", "mujer", vars_x, bloques)

print("=" * 65)
print("DESCOMPOSICIÓN OAXACA-BLINDER POOLED (Neumark)")
print("=" * 65)
print(f"Brecha mensual total (log):  {res['brecha_log']:+.4f}")
print(f"                              ({(np.exp(res['brecha_log'])-1)*100:+.1f}% en niveles)")
print(f"\nDe esa brecha:")
print(f"  Parte EXPLICADA:           {res['explicado']:+.4f}  ({res['explicado_pct_brecha']:.1f}%)")
print(f"    └── por horas:           {res['aporte_por_bloque']['horas']:+.4f}")
print(f"    └── por composición:     {res['aporte_por_bloque']['composicion']:+.4f}")
print(f"  Parte NO EXPLICADA:        {res['no_explicado']:+.4f}  ({res['no_explicado_pct_brecha']:.1f}%)")
print(f"\nR² del modelo pooled: {res['r2_pool']:.4f}")


## 2. Bootstrap para intervalos de confianza

En cada réplica re-muestreo el dataset con reemplazo, re-estimo los tres modelos (hombres, mujeres, pooled) y vuelvo a descomponer. 240 réplicas dan IC estables al 95%.

> **Aviso de cómputo:** ~0.7 segundos por réplica. 240 réplicas tardan ~3 minutos. Si tu máquina es lenta, baja `n_iter` a 100. Si tienes paciencia, súbelo a 1000.


In [ ]:
# Comentado por defecto — descomenta para correr en serio.
# boot = oaxaca.bootstrap_oaxaca(df, formula, "fac_tri", "mujer", vars_x, bloques,
#                                  n_iter=240, seed=42)
# boot.to_csv(OUTPUTS / "oaxaca_bootstrap.csv", index=False)

# Si ya tienes el bootstrap guardado:
boot = pd.read_csv(OUTPUTS / "oaxaca_bootstrap.csv")
print(f"Réplicas bootstrap cargadas: {len(boot)}")
boot.describe().round(4)


In [ ]:
# Tabla de IC al 95% en porcentaje de la brecha total
componentes = {
    "Brecha total":        boot["brecha_log"],
    "Explicado":           boot["explicado"],
    "  ├ por horas":       boot["aporte_horas"],
    "  └ por composición": boot["aporte_composicion"],
    "No explicado":        boot["no_explicado"],
}
filas = []
for nombre, serie in componentes.items():
    med = serie.median()
    lo, hi = np.percentile(serie, [2.5, 97.5])
    if nombre == "Brecha total":
        filas.append({"Componente": nombre,
                       "Mediana (log)": f"{med:+.4f}",
                       "% de brecha": "—",
                       "IC 95%": f"[{lo:+.4f}, {hi:+.4f}]"})
    else:
        brecha_med = boot["brecha_log"].median()
        pct_med = med / brecha_med * 100
        pct_lo = lo / brecha_med * 100
        pct_hi = hi / brecha_med * 100
        filas.append({"Componente": nombre,
                       "Mediana (log)": f"{med:+.4f}",
                       "% de brecha": f"{pct_med:.1f}%",
                       "IC 95%": f"[{pct_lo:.1f}%, {pct_hi:.1f}%]"})

pd.DataFrame(filas).set_index("Componente")


**El número titular del proyecto:** la mitad de la brecha mensual se explica por horas trabajadas (≈49%) y la otra mitad no se explica con variables observables (≈50%). La composición — educación, sector, formalidad, entidad — explica esencialmente cero (su IC incluye al cero).


## 3. Gráfico principal — la descomposición en dos bloques


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
brecha = boot["brecha_log"].median()
horas_pct  = boot["aporte_horas"].median()      / brecha * 100
compo_pct  = boot["aporte_composicion"].median()/ brecha * 100
no_exp_pct = boot["no_explicado"].median()      / brecha * 100

horas_lo, horas_hi = np.percentile(boot["aporte_horas"]/boot["brecha_log"]*100, [2.5,97.5])
no_lo, no_hi       = np.percentile(boot["no_explicado"]/boot["brecha_log"]*100, [2.5,97.5])

izq = 0
for v, c, et, (lo, hi) in [
    (horas_pct,  "#2c7fb8", "Horas trabajadas",              (horas_lo, horas_hi)),
    (compo_pct,  "#a6bddb", "Composición",                   (None,     None)),
    (no_exp_pct, "#d95f0e", "No explicado\n(residual)",      (no_lo,    no_hi)),
]:
    ax.barh(["Brecha mensual"], [v], left=[izq], color=c, edgecolor="white",
            linewidth=2, height=0.5)
    if abs(v) >= 3:
        ax.text(izq+v/2, 0,     f"{v:.1f}%", va="center", ha="center",
                color="white", fontweight="bold", fontsize=14)
        if lo is not None:
            ax.text(izq+v/2, -0.35, f"IC95: [{lo:.1f}, {hi:.1f}]",
                    va="center", ha="center", color="gray", fontsize=8.5)
        ax.text(izq+v/2, 0.45, et, va="center", ha="center", fontsize=10.5,
                fontweight="bold", color=c)
    izq += v

ax.set_xlim(-3, 105)
ax.set_xticks([]); ax.set_yticks([])
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_visible(False)

ax.text(50, 1.05, "De la brecha mensual del 32%, la mitad la explican las horas trabajadas.\n"
        "La otra mitad no se explica con variables observables.",
        ha="center", fontsize=13, fontweight="bold", transform=ax.transData)
ax.text(50, -0.85, "Descomposición Oaxaca-Blinder pooled (Neumark) sobre log(ingreso mensual). "
        "IC 95% por bootstrap (240 réplicas).\nLa composición incluye escolaridad, experiencia, "
        "sector, condición de subordinación y entidad federativa.",
        ha="center", fontsize=8.5, color="gray", transform=ax.transData)
plt.tight_layout()
plt.savefig(OUTPUTS / "oaxaca_g1_descomposicion.png")
plt.show()


## 4. Distribuciones bootstrap por componente

Permite ver no solo el punto estimado sino la incertidumbre. Si una distribución cruza el cero, esa contribución no es estadísticamente distinta de cero.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
configs = [
    ("aporte_horas",       "Aporte de horas",        "#2c7fb8", axes[0]),
    ("aporte_composicion", "Aporte de composición",  "#a6bddb", axes[1]),
    ("no_explicado",       "No explicado (residual)","#d95f0e", axes[2]),
]
for col, titulo, color, ax in configs:
    pcts = boot[col] / boot["brecha_log"] * 100
    ax.hist(pcts, bins=25, color=color, alpha=0.85, edgecolor="white")
    med, lo, hi = pcts.median(), np.percentile(pcts, 2.5), np.percentile(pcts, 97.5)
    ax.axvline(med, color="black", linewidth=1.5)
    ax.axvline(lo,  color="black", linestyle="--", linewidth=1, alpha=0.6)
    ax.axvline(hi,  color="black", linestyle="--", linewidth=1, alpha=0.6)
    ax.axvline(0,   color="red",   linewidth=0.8, alpha=0.5)
    ax.set_title(f"{titulo}\nMediana: {med:.1f}%  |  IC95: [{lo:.1f}, {hi:.1f}]", fontsize=11)
    ax.set_xlabel("% de la brecha total")

axes[0].set_ylabel("Frecuencia (240 réplicas)")
plt.suptitle("Distribución bootstrap de los componentes de la brecha",
             fontsize=12.5, fontweight="bold", y=1.04)
plt.tight_layout()
plt.savefig(OUTPUTS / "oaxaca_g2_distribuciones.png")
plt.show()


## Debrief del notebook 04

**Lo que aprendí:**

La descomposición da un resultado más limpio del que esperaba. Las dos hipótesis principales del plan se contestaron con datos:

- **H1 (horas explican mucho de la brecha):** confirmada. Horas aportan 49.1% (IC 95%: 47–52%) de la brecha mensual.
- **H2 (composición aporta algo):** rechazada. La composición observable aporta 0.5% (IC: −2.1%, 3.0%), no es distinta de cero.
- **H3 (residual entre 3 y 6 puntos):** subestimada. El residual es enorme: 50.4% de la brecha, equivalente a 16% de salario adicional que no se explica.

**Lo que se me resistió:**

- Statsmodels formula no se llevaba bien con pickle entre procesos. Reconstruir los modelos en cada réplica del bootstrap fue la decisión correcta — más lento pero limpio.
- El bootstrap completo tarda ~3 minutos para 240 réplicas. Para presentaciones rápidas conviene bajar a 100; para la versión publicable, 1000.

**Para los notebooks siguientes:**

- **Notebook 05 (Heckman):** corregir por sesgo de selección de mujeres ocupadas. La participación laboral femenina (~46%) es selectiva — las que sí participan pueden ser una muestra "alta" en variables no observadas. Si Heckman baja la brecha residual significativamente, hay que decirlo; si no la mueve, el residual del 50% es robusto.
- **Notebook 06 (visualizaciones finales):** unir las gráficas en un set coherente para el README y el carrusel de LinkedIn. La narrativa: "El 32% de brecha mensual = 49% horas + 50% no explicado. Las mujeres ocupadas tienen MÁS educación y MÁS formalidad. La explicación obvia no aplica."

**Limitaciones honestas:**

- "No explicado" no es "discriminación" estrictamente. Puede incluir habilidades no observadas, preferencias por puestos compatibles con cuidados, redes profesionales, calidad de la educación. Es un **techo** de la discriminación pura.
- Variables omitidas potencialmente importantes: tamaño de empresa, antigüedad en el puesto, presencia de hijos. Algunas las podríamos recuperar pegando COE1 (lo dejamos para el notebook 05).
